1. Define Schema

In [0]:
file_schema = """
id int,
name string,
dop string,
phone long,
amount string,
discount string
"""

In [0]:
sales_df = spark.read\
        .format('csv')\
            .schema(file_schema)\
                .option('header','true')\
                    .load('/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv')

In [0]:
sales_df.display()

In [0]:
sales_df.describe().display()


1.4 List down the problems you want to fix
1. Convert id from integer to string and rename it as transaction_id.
2. Rename the name column to customer_name.
3. Convert the dop to date format and rename the column to date_of_purchase.
4. Rename the phone column to customer_phone
5. Convert the amount to a long value and filter out nulls and outlier values. 
6. Rename the column to purchase_amount
7. Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

1.Renaming multiple columns first.

In [0]:
renamed_columns =( {"id" : "transaction_id",
                    "name" : "customer_name",
                    "dop" : "date_of_purchase",
                    "phone" : "customer_phone",
                    "amount" : "purchase_amount",
                    "discount" : "applied_discount"
                    
                     }
)

In [0]:
sales_df_renamed = sales_df.withColumnsRenamed(renamed_columns)

In [0]:
sales_df_renamed.display()

1. Convert id from integer to string 

In [0]:
from pyspark.sql.functions import expr, cast

sales_df_renamed = sales_df_renamed.withColumn('transaction_id',expr('CAST(transaction_id AS STRING)'))

3. Convert multiple date formats in date of purchase column to standardised date format.

In [0]:
from pyspark.sql.functions import coalesce,to_date,col,try_to_date

date_formats = ["yyyy-MM-dd" , "yyyy-MM-d", "dd-MM-yyyy"]
sales_df_renamed_date_format_standardised =  sales_df_renamed.withColumn('date_of_purchase',\
    (coalesce(*\
            [try_to_date(col('date_of_purchase'), each_date_format) for each_date_format in date_formats])))
    

In [0]:
sales_df_renamed_date_format_standardised.display()

4. Convert the purchase amount to a long value and filter out nulls and outlier values.

In [0]:
sales_df_purchase_amount_format_adjust = sales_df_renamed_date_format_standardised.withColumn('purchase_amount', col('purchase_amount').cast('long'))

In [0]:
sales_df_purchase_amount_format_adjust.display()

In [0]:
#using the .agg approach
#add scalar values first into first, add then to df and then create calcs for outlier detection using filter method

from pyspark.sql.functions import sum, avg, stddev,col

sales_df_sum_purchase_amount = sales_df_purchase_amount_format_adjust.agg(sum(col('purchase_amount')).alias('purchase_amount_total'),\
 avg(col('purchase_amount')).alias('purchase_amount_avg'),
 stddev(col('purchase_amount')).alias('purchase_amount_st_dev') )

In [0]:
sales_df_sum_purchase_amount.display()

#### Extract the scalar values from the result

In [0]:
purchase_amount_avg = sales_df_sum_purchase_amount.collect()[0][0]
purchase_amount_stdev = sales_df_sum_purchase_amount.collect()[0][2]

add the columns to the df .next step would be filtering rows based on keeping rows where purchase amount falls within 1 stdev->.Outlier logic

In [0]:
from pyspark.sql.functions import lit

sales_df_remove_outlier_purchase_amount = sales_df_purchase_amount_format_adjust.withColumns(
    {
     'purchase_amount_avg' : lit(purchase_amount_avg),
     'purchase_amount_stdev' : lit(purchase_amount_stdev),

    }
)

In [0]:
sales_df_remove_outlier_purchase_amount.display()

In [0]:
sales_df_remove_outlier_purchase_amount = (
    sales_df_remove_outlier_purchase_amount.filter(
        col("purchase_amount") <= 1*col("purchase_amount_stdev")
))

In [0]:
sales_df_remove_outlier_purchase_amount.display()

7. Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

In [0]:
from pyspark.sql.functions import when

sales_df_discount_cleansing = sales_df_remove_outlier_purchase_amount.withColumn(\
   "applied_discount", \
        when(col("applied_discount") =="nil" ,None)\
            .otherwise(col("applied_discount")).cast("double")   # if false
)

In [0]:
sales_df_discount_cleansing.display()